# Tutorial: Palette catalog summaries

Audience:
- Developers who want to summarize a CSV catalog of color palettes using only the Python standard library.

Prerequisites:
- Basic Python (functions, dictionaries, lists).
- Familiarity with reading CSV files.

Learning goals:
- By the end, the reader can group CSV rows by a normalized key and compute per-group summaries using `csv` and `collections` from the standard library.


## Outline

1. Setup
2. Define `summarize_palettes`
3. Run it on the catalog
4. Exercises


In [ ]:
# Setup cell: keep it short and deterministic
from __future__ import annotations

import csv
from collections import defaultdict

CATALOG_PATH = "/inputs/colors.csv"
CATALOG_PATH


## Step 1 - Group rows by normalized primary color

`summarize_palettes(csv_path)` reads the CSV using its header row, strips whitespace from
`Primary (Hex)` and `Product Type`, uppercases the primary color, and groups rows by that
normalized primary color. For each group it returns the normalized primary color, the row
count (including duplicates), and the sorted unique product types. Groups are ordered by
count descending, then primary ascending.


In [ ]:
def summarize_palettes(csv_path: str) -> list[dict]:
    """Group a palette catalog CSV by normalized primary color.

    Reads the CSV using its header row (column order and extra columns do not
    matter), strips surrounding whitespace from ``Primary (Hex)`` and
    ``Product Type``, uppercases the primary color, and groups rows by that
    normalized primary color.
    """
    groups: dict[str, dict] = defaultdict(lambda: {"count": 0, "product_types": set()})

    with open(csv_path, "r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            primary = row["Primary (Hex)"].strip().upper()
            product_type = row["Product Type"].strip()
            group = groups[primary]
            group["count"] += 1
            group["product_types"].add(product_type)

    summaries = [
        {
            "primary": primary,
            "count": data["count"],
            "product_types": sorted(data["product_types"]),
        }
        for primary, data in groups.items()
    ]
    summaries.sort(key=lambda g: (-g["count"], g["primary"]))
    return summaries


summarize_palettes


## Step 2 - Run it on the catalog

Call `summarize_palettes` on the bundled catalog and inspect the first few groups.


In [ ]:
palette_summary = summarize_palettes(CATALOG_PATH)
len(palette_summary), palette_summary[:3]


## Exercises

- Predict how many groups `summarize_palettes` will return before running the previous cell.
- Try the function on a CSV with only a header row and confirm it returns an empty list.
- Try the function on rows with lowercase hex values and duplicated rows; confirm counts and
  product types are still correct.
- Try a CSV where columns are reordered compared to the original; confirm the result is
  unchanged, since lookups use header names rather than column positions.


In [ ]:
# Exercise scaffold: build a small in-memory CSV to explore summarize_palettes
import io
import tempfile
import os

sample_csv = (
    "Product Type,Primary (Hex)\n"
    "Alpha,#2563eb\n"
    "  Beta  ,  #2563EB  \n"
    '"Gamma, Inc",#FF0000\n'
)

fd, sample_path = tempfile.mkstemp(suffix=".csv")
with os.fdopen(fd, "w", encoding="utf-8", newline="") as handle:
    handle.write(sample_csv)

sample_summary = summarize_palettes(sample_path)
os.remove(sample_path)
sample_summary
